# Notebook 11 — Hyperparameter Tuning (Corrected)

## AI-Supply-Chain-Digital-Marketing

### Objective
Tune the selected **XGBoost** model while preserving the model-selection rule.

The tuning process uses only the **training split** for cross-validation. The separate validation split is used only to compare the tuned candidate with the Notebook 10 baseline.

### Important decision rule
Because tuning is not guaranteed to improve the model, this notebook does **not** automatically replace the baseline.

- Primary metric: validation **F1**
- If tuned validation F1 > baseline validation F1 → tuned XGBoost becomes the final candidate.
- Otherwise → baseline XGBoost remains the final candidate.
- Test data is never used for this decision.

This prevents a tuned model with lower F1 from incorrectly replacing the stronger baseline.


In [17]:
from pathlib import Path
import json
import time
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)
from xgboost import XGBClassifier

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models" / "supply_chain"
RESULTS_DIR = BASE_DIR / "results" / "metrics"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR.resolve())
print("Processed directory exists:", PROCESSED_DIR.exists())
print("Model directory exists:", MODEL_DIR.exists())


Base directory: C:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1
Processed directory exists: True
Model directory exists: True


## Cell 3 — Load train, validation and test data

In [18]:
X_train = pd.read_csv(PROCESSED_DIR / "X_train_preprocessed.csv")
y_train = pd.read_csv(PROCESSED_DIR / "y_train_preprocessed.csv").squeeze("columns")

X_validation = pd.read_csv(PROCESSED_DIR / "X_validation_preprocessed.csv")
y_validation = pd.read_csv(PROCESSED_DIR / "y_validation_preprocessed.csv").squeeze("columns")

X_test = pd.read_csv(PROCESSED_DIR / "X_test_preprocessed.csv")
y_test = pd.read_csv(PROCESSED_DIR / "y_test_preprocessed.csv").squeeze("columns")

assert X_train.shape == (3500, 590)
assert X_validation.shape == (750, 590)
assert X_test.shape == (750, 590)

assert len(y_train) == 3500
assert len(y_validation) == 750
assert len(y_test) == 750

assert list(X_train.columns) == list(X_validation.columns) == list(X_test.columns)

assert not X_train.isna().any().any()
assert not X_validation.isna().any().any()
assert not X_test.isna().any().any()

assert np.isfinite(X_train.to_numpy(dtype=float)).all()
assert np.isfinite(X_validation.to_numpy(dtype=float)).all()
assert np.isfinite(X_test.to_numpy(dtype=float)).all()

print("Input validation: PASS")
print("Training:", X_train.shape)
print("Validation:", X_validation.shape)
print("Test:", X_test.shape)


Input validation: PASS
Training: (3500, 590)
Validation: (750, 590)
Test: (750, 590)


## Cell 4 — Verify XGBoost selection from Notebook 09 and baseline from Notebook 10

In [19]:
selection_path = MODEL_DIR / "all_four_model_selection_info.json"
baseline_model_path = MODEL_DIR / "trained_candidate_model.joblib"
baseline_metadata_path = MODEL_DIR / "training_metadata.json"

assert selection_path.exists()
assert baseline_model_path.exists()
assert baseline_metadata_path.exists()

with open(selection_path, "r", encoding="utf-8") as f:
    selection_info = json.load(f)

with open(baseline_metadata_path, "r", encoding="utf-8") as f:
    baseline_metadata = json.load(f)

assert selection_info["selected_model"] == "XGBoost"
assert selection_info["selection_metric"] == "F1"
assert selection_info["selection_split"] == "validation"

assert baseline_metadata["selected_model"] == "XGBoost"
assert baseline_metadata["test_used_for_training"] is False
assert baseline_metadata["test_used_for_selection"] is False
assert baseline_metadata["test_prediction_generated"] is False

baseline_model = joblib.load(baseline_model_path)
assert isinstance(baseline_model, XGBClassifier)

print("Notebook 09 selection: XGBoost")
print("Notebook 10 baseline: XGBoost")
print("Baseline artifact verification: PASS")


Notebook 09 selection: XGBoost
Notebook 10 baseline: XGBoost
Baseline artifact verification: PASS


## Cell 5 — Define the baseline and conservative tuning search space

In [20]:
base_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

param_distributions = {
    "n_estimators": [200, 300, 400, 500],
    "max_depth": [4, 5, 6, 7],
    "learning_rate": [0.03, 0.05, 0.08, 0.1],
    "subsample": [0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.2, 0.5],
    "reg_alpha": [0, 0.01, 0.1],
    "reg_lambda": [1.0, 2.0, 5.0]
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

tuning_search = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_distributions,
    n_iter=12,
    scoring="f1",
    n_jobs=-1,
    cv=cv,
    refit=True,
    random_state=42,
    verbose=1,
    return_train_score=True
)

print("XGBoost tuning configuration: PASS")
print("Iterations:", 12)
print("CV folds:", 3)
print("Scoring:", "F1")


XGBoost tuning configuration: PASS
Iterations: 12
CV folds: 3
Scoring: F1


## Cell 6 — Run tuning using training data only

In [21]:
tuning_start = time.perf_counter()

# IMPORTANT: validation and test data are not supplied here.
tuning_search.fit(X_train, y_train)

tuning_seconds = time.perf_counter() - tuning_start

tuned_model = tuning_search.best_estimator_
best_cv_f1 = float(tuning_search.best_score_)
best_params = tuning_search.best_params_

assert isinstance(tuned_model, XGBClassifier)

print("Tuning completed.")
print("Best CV F1:", round(best_cv_f1, 6))
print("Best parameters:")
print(best_params)
print("Tuning time (seconds):", round(tuning_seconds, 4))


Fitting 3 folds for each of 12 candidates, totalling 36 fits
Tuning completed.
Best CV F1: 0.766471
Best parameters:
{'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.03, 'gamma': 0.5, 'colsample_bytree': 0.7}
Tuning time (seconds): 11.967


## Cell 7 — Evaluate baseline and tuned models on validation

In [22]:
def evaluate_model(model, X, y, name):
    pred = model.predict(X)
    prob = model.predict_proba(X)[:, 1]

    return {
        "Model": name,
        "Accuracy": accuracy_score(y, pred),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y, prob),
        "PR_AUC": average_precision_score(y, prob)
    }

baseline_validation = evaluate_model(
    baseline_model, X_validation, y_validation, "Baseline XGBoost"
)

tuned_validation = evaluate_model(
    tuned_model, X_validation, y_validation, "Tuned XGBoost"
)

comparison_df = pd.DataFrame([
    baseline_validation,
    tuned_validation
])

display(comparison_df.round(6))

baseline_f1 = float(
    comparison_df.loc[
        comparison_df["Model"] == "Baseline XGBoost", "F1"
    ].iloc[0]
)

tuned_f1 = float(
    comparison_df.loc[
        comparison_df["Model"] == "Tuned XGBoost", "F1"
    ].iloc[0]
)

print("Baseline validation F1:", round(baseline_f1, 6))
print("Tuned validation F1:", round(tuned_f1, 6))


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Baseline XGBoost,0.766667,0.799569,0.818985,0.809160,0.839937,0.904508
1,Tuned XGBoost,0.762667,0.798265,0.812362,0.805252,0.843059,0.905292


Baseline validation F1: 0.80916
Tuned validation F1: 0.805252


## Cell 8 — Apply the project model-retention rule

The tuned model is retained **only if it improves validation F1**.

This is an actual comparison generated from the current run; no result is hard-coded.


In [23]:
if tuned_f1 > baseline_f1:
    final_candidate_name = "Tuned XGBoost"
    final_candidate_model = tuned_model
    tuning_improved_f1 = True
else:
    final_candidate_name = "Baseline XGBoost"
    final_candidate_model = baseline_model
    tuning_improved_f1 = False

f1_change = tuned_f1 - baseline_f1

print("Baseline F1:", round(baseline_f1, 6))
print("Tuned F1:", round(tuned_f1, 6))
print("F1 change:", round(f1_change, 6))
print("Tuning improved F1:", tuning_improved_f1)
print("Final candidate:", final_candidate_name)

if tuning_improved_f1:
    print("Decision: Tuned XGBoost retained.")
else:
    print("Decision: Baseline XGBoost retained because tuning did not improve F1.")


Baseline F1: 0.80916
Tuned F1: 0.805252
F1 change: -0.003909
Tuning improved F1: False
Final candidate: Baseline XGBoost
Decision: Baseline XGBoost retained because tuning did not improve F1.


## Cell 9 — Save tuning comparison results

In [24]:
comparison_path = RESULTS_DIR / "xgboost_baseline_vs_tuned_validation.csv"
comparison_df.to_csv(comparison_path, index=False)

cv_results_path = RESULTS_DIR / "xgboost_hyperparameter_tuning_results.csv"

cv_results_df = pd.DataFrame(tuning_search.cv_results_)

keep_columns = [
    "rank_test_score",
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "std_train_score",
    "params"
]

keep_columns = [c for c in keep_columns if c in cv_results_df.columns]
cv_results_df[keep_columns].sort_values(
    "rank_test_score"
).to_csv(cv_results_path, index=False)

print("Validation comparison saved:", comparison_path.exists())
print("CV results saved:", cv_results_path.exists())


Validation comparison saved: True
CV results saved: True


## Cell 10 — Save tuning metadata and decision

In [25]:
tuning_info_path = MODEL_DIR / "xgboost_tuning_info.json"

def json_safe(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    return value

safe_best_params = {
    key: json_safe(value)
    for key, value in best_params.items()
}

tuning_info = {
    "model": "XGBoost",
    "tuning_metric": "F1",
    "tuning_data": "training",
    "cross_validation": "StratifiedKFold",
    "cv_folds": 3,
    "random_search_iterations": 12,
    "random_state": 42,
    "best_cv_f1": best_cv_f1,
    "best_params": safe_best_params,
    "baseline_validation_f1": baseline_f1,
    "tuned_validation_f1": tuned_f1,
    "validation_f1_change": f1_change,
    "tuning_improved_f1": tuning_improved_f1,
    "final_candidate": final_candidate_name,
    "test_used_for_tuning": False,
    "test_used_for_selection": False,
    "test_prediction_generated": False
}

with open(tuning_info_path, "w", encoding="utf-8") as f:
    json.dump(tuning_info, f, indent=2)

print("Tuning metadata saved:", tuning_info_path.exists())


Tuning metadata saved: True


## Cell 11 — Save the final candidate for Notebook 12

If tuning improved validation F1, the tuned model is saved.

If tuning did not improve F1, the original baseline XGBoost is retained.

The test set is still untouched.


In [26]:
final_candidate_path = MODEL_DIR / "final_xgboost_candidate.joblib"

joblib.dump(final_candidate_model, final_candidate_path)

assert final_candidate_path.exists()

print("Final candidate:", final_candidate_name)
print("Final candidate path:", final_candidate_path)
print("Final candidate exists:", final_candidate_path.exists())


Final candidate: Baseline XGBoost
Final candidate path: c:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\models\supply_chain\final_xgboost_candidate.joblib
Final candidate exists: True


## Cell 12 — Reload and verify final candidate

In [27]:
reloaded_final_candidate = joblib.load(final_candidate_path)

assert isinstance(reloaded_final_candidate, XGBClassifier)

reloaded_predictions = reloaded_final_candidate.predict(X_validation)

assert len(reloaded_predictions) == 750
assert set(np.unique(reloaded_predictions)).issubset({0, 1})

print("Reloaded candidate type:", type(reloaded_final_candidate).__name__)
print("Validation predictions after reload:", len(reloaded_predictions))
print("Reload verification: PASS")


Reloaded candidate type: XGBClassifier
Validation predictions after reload: 750
Reload verification: PASS


## Cell 13 — Explicit test-set protection

In [28]:
test_used_for_tuning = False
test_used_for_selection = False
test_prediction_generated = False

assert test_used_for_tuning is False
assert test_used_for_selection is False
assert test_prediction_generated is False

print("Test used for tuning: NO")
print("Test used for selection: NO")
print("Test prediction generated: NO")


Test used for tuning: NO
Test used for selection: NO
Test prediction generated: NO


## Cell 14 — Final validation

In [29]:
assert selection_info["selected_model"] == "XGBoost"

assert isinstance(tuned_model, XGBClassifier)
assert isinstance(final_candidate_model, XGBClassifier)
assert isinstance(reloaded_final_candidate, XGBClassifier)

assert X_train.shape == (3500, 590)
assert X_validation.shape == (750, 590)
assert X_test.shape == (750, 590)

assert len(reloaded_predictions) == 750

assert comparison_path.exists()
assert cv_results_path.exists()
assert tuning_info_path.exists()
assert final_candidate_path.exists()

assert tuning_info["model"] == "XGBoost"
assert tuning_info["tuning_metric"] == "F1"
assert tuning_info["tuning_data"] == "training"

assert tuning_info["test_used_for_tuning"] is False
assert tuning_info["test_used_for_selection"] is False
assert tuning_info["test_prediction_generated"] is False

assert final_candidate_name in {
    "Baseline XGBoost",
    "Tuned XGBoost"
}

assert np.isfinite(best_cv_f1)
assert np.isfinite(baseline_f1)
assert np.isfinite(tuned_f1)

print("Tuned model: XGBoost")
print("Training rows:", X_train.shape[0])
print("Training features:", X_train.shape[1])
print("Validation rows:", X_validation.shape[0])
print("Test rows:", X_test.shape[0])
print("Baseline validation F1:", round(baseline_f1, 6))
print("Tuned validation F1:", round(tuned_f1, 6))
print("F1 change:", round(f1_change, 6))
print("Tuning improved F1:", tuning_improved_f1)
print("Final candidate:", final_candidate_name)
print("Test used for tuning: False")
print("Test used for selection: False")
print("Test prediction generated: False")
print("Final candidate exists:", final_candidate_path.exists())

print("\nNotebook 11 final validation: PASS")


Tuned model: XGBoost
Training rows: 3500
Training features: 590
Validation rows: 750
Test rows: 750
Baseline validation F1: 0.80916
Tuned validation F1: 0.805252
F1 change: -0.003909
Tuning improved F1: False
Final candidate: Baseline XGBoost
Test used for tuning: False
Test used for selection: False
Test prediction generated: False
Final candidate exists: True

Notebook 11 final validation: PASS
